In [ ]:
abc = 'abc'
xyz = 'xyz'
n_jobs = 1
iter_cv = 100
nrand = 10

In [ ]:
pip install neureval==1.1.1

In [ ]:
%%bash
dx cd /prove_cohorts/prova_clustering_MDD_FAST_DKT_FA/neureval_MDD_RDS-4dep/new_analysis/DKT_cortical_thickness_only/training_55_test_45

In [ ]:
%%bash
dx download data_55_training_DKT_thickness.csv

In [ ]:
%%bash
dx download covariates_55_training_DKT_thickness.csv

In [ ]:
%%bash
dx download data_all_DKT_thickness.csv

In [ ]:
%%bash
dx download covariates_all_DKT_thickness.csv

In [ ]:
# Make the required imports
import pandas as pd
import numpy as np
from neureval.best_nclust_cv_confounds import FindBestClustCVConfounds
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.metrics import zero_one_loss, adjusted_mutual_info_score, silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import pickle as pkl
import logging
import sys
from neureval.visualization import plot_metrics
from neureval.internal_baselines_confounds import select_best
from neureval.utils import kuhn_munkres_algorithm

In [ ]:
# Define working directories and create a new folder called 'models'
main_path = '/opt/notebooks'
data_path = main_path
os.chdir(data_path)

In [ ]:
# Import data and covariates files
#database = 'Participant_table_MDD_FAST_50.csv'
#covariates_file = 'Covariates_MDD_FAST_50.csv'
# If there are multiple sheets, specify the name of the current sheet 
data = pd.read_csv('/opt/notebooks/data_55_training_DKT_thickness.csv', delimiter=';')
cov = pd.read_csv('/opt/notebooks/covariates_55_training_DKT_thickness.csv', delimiter=';')

In [ ]:
# Define two dictionaries for modalities (i.e., views in multiview clustering) and covariates variables.
# For each kind of modality, specify the indexes of the columuns related to the features to be used for clustering
# You can also specify different covaiates for each kind of modality
modalities = {'gm_thickness': data.iloc[:,1:]}
covariates = {'gm_thickness': cov.iloc[:,1:-1]}

In [ ]:
# Define multiview clusteirng and classifier algortihms with optimized parameters
c = KMeans(init='k-means++',random_state=42)
s = SVC(C=0.1, kernel='linear', random_state=42)

In [ ]:
# Initialize FindBestClustCVMultiview class. It performs (repeated) k-folds cross-validation to select the best number of clusters.
# Parameters to be specified:
# nfold: cross-validation folds
# nrand: number of random labelling iterations, default 10
# n_jobs: number of jobs to run in parallel, default (number of cpus - 1)
# clust_range: list with number of clusters (e.g., list(range(2,3))), default None
findbestclust = FindBestClustCVConfounds(c,s, preprocessing=None, nfold=2, nrand=nrand,  n_jobs=n_jobs, nclust_range=list(range(2,5)))

In [ ]:
# Run FindBestClustCVMultiview. It returns normlaized stability (metrics), best number of clusters (bestncl), and clusters' labels (tr_lab).
# Parameters to be specified:
# iter_cv: number of repeated cross-validation, default 1
# strat: stratification vector for cross-validation splits, default None
metrics, bestncl, tr_lab = findbestclust.best_nclust_confounds(data, modalities, covariates, iter_cv=iter_cv, strat_vect=None)
val_results = list(metrics['val'].values())
val_results = np.array(val_results, dtype=object)

print(f"Best number of clusters: {bestncl}")
print(f"Validation set normalized stability (misclassification): {metrics['val'][bestncl]}")
print(f"Result accuracy (on test set): "
      f"{1-val_results[0,0]}")

In [ ]:
# Normalized stability plot. For each number of clusters, normalized stabilities are represented for both training (dashed line) and validation sets (continuous line).
# Colors for training and validation sets can be changed (default: ('black', 'black')).
# To save the plot, specify the file name for saving figure in png format.
plot = plot_metrics(metrics, color=('black', 'black'), save_fig='plot_55_training_DKT.png')

In [ ]:
# SILHOUETTE SCORE
# Use select_best to calcluate silhouette score. Specify silhouette_score as int_measure and 'max' as select parameter.
# It returns silhouette score (sil_score), number of clusters selected (sil_best), and silhouette labels (sil_labels)
logging.info("Silhouette score based selection")
sil_score, sil_best, sil_label = select_best(data, modalities, covariates, c, silhouette_score, preprocessing=None, select='max', nclust_range=list(range(2,5)))

logging.info(f"Best number of clusters (and scores): "
             f"{{{sil_best}({sil_score})}}")
logging.info(f'AMI (true labels vs clustering labels) training = '
             f'{adjusted_mutual_info_score(tr_lab, kuhn_munkres_algorithm(np.int32(tr_lab), np.int32(sil_label)))}')
logging.info('\n\n')

# Save results obtained with silhouette score.
# Silhouette score, number of clusters, and clusters' labels are organized as a dictionary and saved as a pickle object
sil_results = {'sil_score': sil_score,
               'sil_best': sil_best,
               'sil_label': sil_label}
pkl.dump(sil_results, open('./sil_results_55_training_DKT.pkl', 'wb'))
# To open the results, uncomment the following line
# pd.read_pickle('./sil_results_model_name.pkl')

In [ ]:
# DAVIES-BOULDIN SCORE
# Use select_best to calcluate Davies-Bouldin score. Specify davies_bouldin_score as int_measure and 'min' as select parameter.
# It returns davies-bouldin score (db_score), number of clusters selected (db_best), and davies-bouldin labels (db_labels)
logging.info("Davies-Bouldin score based selection")
db_score, db_best, db_label = select_best(data,modalities, covariates, c, davies_bouldin_score,preprocessing=None,
                                                   select='min', nclust_range=list(range(2,5)))

logging.info(f"Best number of clusters (and scores): "
             f"{{{db_best}({db_score})}}")
logging.info(f'AMI (true labels vs clustering labels) training = '
             f'{adjusted_mutual_info_score(tr_lab, kuhn_munkres_algorithm(np.int32(tr_lab), np.int32(db_label)))}')
logging.info('\n\n')

# Save results obtained with Davies-Bouldin score.
# Davies-Bouldin score, number of clusters, and clusters' labels are organized as a dictionary and saved as a pickle object
db_results = {'db_score': db_score,
               'db_best': db_best,
               'db_label': db_label}
pkl.dump(db_results, open('./db_results_55_training_DKT.pkl', 'wb'))
# To open the results, uncomment the following line
# pd.read_pickle('./db_results_model_name.pkl')

In [ ]:
## EXTERNAL VALIDATION
# Import data and covariates files
#database = 'Participant_table_MDD_FAST_50.csv'
#covariates_file = 'Covariates_MDD_FAST_50.csv'
# If there are multiple sheets, specify the name of the current sheet 
data_all = pd.read_csv('/opt/notebooks/data_all_DKT_thickness.csv', delimiter=';')
cov_all = pd.read_csv('/opt/notebooks/covariates_all_DKT_thickness.csv', delimiter=';')
data_all = data_all.drop(['train_test'],axis=1)
cov_all = cov_all.drop(['train_test'],axis=1)

In [ ]:
# Specify the number of subjects belonging to the training set
training_dim = 842
tr_idx = list(range(0,training_dim))
val_idx = list(range(training_dim,len(data_all)))

print(f"Training set sample size: {len(tr_idx)}")
print(f"Validation set sample size: {len(val_idx)}")

In [ ]:
# Define two dictionaries for modalities (i.e., views in multiview clustering) and covariates variables.
# For each kind of modality, specify the indexes of the columuns related to the features to be used for clustering
# You can also specify different covaiates for each kind of modality
modalities_val = {'gm_thickness': data_all.iloc[:,1:]}
covariates_val = {'gm_thickness': cov_all.iloc[:,1:-1]}

In [ ]:
# Validate the identified clusters
# Parameters to be specified:
# data: dataset
# modalities: dictionary specifying the features for each modality
# covariates: dictionary specifying the covariates for each modality
# tr_idx: number of training samples
# val_idx: number of validation samples
# nclust: specify the best number of clusters idenfied in the training dataset (i.e., bestncl)
out = findbestclust.evaluate_confounds(data_all, modalities_val, covariates_val, tr_idx, val_idx, nclust=bestncl, tr_lab=None)
print(f"Training ACC: {out.train_acc}, Test ACC: {out.test_acc}")

In [ ]:
# Remove dataset and covariates .csv files
files_in_directory = os.listdir(main_path)
filtered_files = [file for file in files_in_directory if file.endswith('.csv')]
for file in filtered_files:
    path_to_file = os.path.join(main_path, file)
    os.remove(path_to_file)

In [ ]:
# Save database with cluster labels obtained from the training set for post-hoc analyses
labels_tr = pd.DataFrame(tr_lab, columns=['Labels'])
data_all_tr = pd.concat([data, labels_tr], axis=1)
data_all_tr.to_csv('Labels_findbestclust_training_55_DKT.csv', index=True)

In [ ]:
# Save cluster labels in the training set
labels_val = pd.DataFrame(out.train_cllab, columns=['labels'])
labels_val.to_csv('Labels_training_55_DKT.csv', index=True)

In [ ]:
# Save cluster labels in the validation set
labels_val = pd.DataFrame(out.test_cllab, columns=['Labels'])
labels_val.to_csv('Labels_test_45_DKT.csv', index=True)

In [ ]:
# Save the findbestclust object to a pickle file for future analyses
pkl.dump(findbestclust, open('./findbestclust_55_training_45_test_DKT.pkl', 'wb'))